In [ ]:
import os
import sys 
print("Active Environment Path:", sys.executable)
from pathlib import Path
import json
import subprocess
import importlib
import tempfile

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, PrecisionRecallDisplay
import shapely

import pystac_client
import planetary_computer
import odc.stac
import rasterio.features

# local
os.chdir("/home/azureuser/EBRD_Tillage/AK_Experiments/AK_pipeline")
sys.path.append(os.getcwd())
from Generic.addSentinelIndices import calculateSentinelIndex, SENTINEL_INDEX_DEPENDENCIES
from utils.classify_tillage import classify_tillage

Active Environment Path: /anaconda/envs/geospatial-base/bin/python


In [ ]:
# ==============================================================================
# 1. HELPER FUNCTIONS FOR IN-MEMORY STAC & RASTER REDUCTION
# ==============================================================================

# This needn't be called for every fields, I am calling it for a sample field. 
def get_utm_crs(gdf: gpd.GeoDataFrame) -> str:
    """Determine the local UTM EPSG code from a GeoDataFrame centroid."""
    centroid = gdf.to_crs("EPSG:4326").union_all().centroid
    lon, lat = centroid.x, centroid.y
    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"

#########################################################33
def compute_event_band_in_memory_s1(
        row: pd.Series,
        catalog_client: pystac_client.Client,
        bands :list,
        temp_aggregations :list,
        spat_aggregations:list,
            
        buffer_days: int = 10,
        resolution: float = 10.0,
        orbit_direction: str = None,  # Optional: "ascending" or "descending"
):
    """Stream Sentinel-1 RTC backscatter bands (VV, VH) for a single event

    directly into memory.
    """
    geom_4326 = row.geometry

    # 1. Date window with buffer
    start_dt = pd.to_datetime(row["end_date"]).strftime("%Y-%m-%d")
    end_dt = (
        pd.to_datetime(row["end_date"]) + pd.Timedelta(days=buffer_days)
    ).strftime("%Y-%m-%d")

    # 2. Prepare STAC query filters (No cloud cover filter for SAR)
    query_params = {}

    if orbit_direction in ["ascending", "descending"]:
        query_params["sat:orbit_state"] = {"eq": orbit_direction}

    # 3. STAC Search directly in memory
    search = catalog_client.search(
        collections=["sentinel-1-rtc"],
        intersects=geom_4326.__geo_interface__,
        datetime=f"{start_dt}/{end_dt}",
        query=query_params if query_params else None,
        limit=100,
    )
    items = list(search.items())

    if not items:
        return np.nan

    # Sign asset URLs for Planetary Computer Azure SAS authentication
    items = [planetary_computer.sign(item) for item in items]

    # 4. Project polygon to local UTM for accurate 10m spatial rasterization
    event_gdf = gpd.GeoDataFrame([row], crs="EPSG:4326")
    utm_crs = get_utm_crs(event_gdf)
    event_utm = event_gdf.to_crs(utm_crs)
    minx, miny, maxx, maxy = event_utm.total_bounds

    # 5. Load Sentinel-1 backscatter assets into memory using odc.stac
    xarray_ds = odc.stac.load(
        items,
        bands=bands,  # Usually ['vv', 'vh']
        crs=utm_crs,
        resolution=resolution,
        x=(minx - 20, maxx + 20),
        y=(miny - 20, maxy + 20),
        chunks={},
    ).load()  # Materialize arrays into RAM

    if ds.sizes.get("time", 0) == 0:
        return np.nan

    # Convert linear amplitude values to float32
    for pol in polarizations:
        ds[pol] = ds[pol].astype("float32")

    return ds

In [ ]:
def get_sentinel2_STAC(
        catalog_client: pystac_client.Client, 
        geometry: shapely.geometry, 
        
)

1. Aggregations need to happen for raw bands too


In [ ]:

def compute_event_band_in_memory( 
    row: pd.Series,
    catalog_client: pystac_client.Client,
    bands :list,
    temp_aggregations :list,
    spat_aggregations:list,
    
    buffer_days: int = 10,
    max_cloud_cover: int = 90,
    resolution: float = 10.0,
):
    """Stream optical bands for a single event directly into memory and return
        Object stored as Xarray Dataset. 
        Xarray is 3 dimentions with x y and time, with '4th dimension' being bands. 
        Index calculation is point wise, which means it only adds to the bands. 
        The bands (data_vars) are callable with names, so the function behaves exactly like a dataframe. 
    
        The pipeline of 1.Temporal Aggregation 2. Rasterization and 3. Spatial Aggregation is best. 
    """
    geom_4326 = row.geometry
    
    # 1. Date window with buffer
    #start_dt = (pd.to_datetime(row["start_date"]) - pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
    start_dt = pd.to_datetime(row["end_date"]).strftime("%Y-%m-%d")
    end_dt = (pd.to_datetime(row["end_date"]) + pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
    
    # 2. STAC Search directly in memory
    search = catalog_client.search(
        collections=["sentinel-2-l2a"],
        intersects=geom_4326.__geo_interface__,
        datetime=f"{start_dt}/{end_dt}",
        query={"eo:cloud_cover": {"lte": max_cloud_cover}},
        limit=100
    )
    items = list(search.items())
    
    if not items:
        return np.nan

    # Sign asset URLs for Planetary Computer Azure SAS authentication
    items = [planetary_computer.sign(item) for item in items]
    
    # 3. Project polygon to local UTM for accurate 10m spatial rasterization
    event_gdf = gpd.GeoDataFrame([row], crs="EPSG:4326")
    utm_crs = get_utm_crs(event_gdf)
    event_utm = event_gdf.to_crs(utm_crs)
    minx, miny, maxx, maxy = event_utm.total_bounds
    
    # 4. Calculate required bands and stream  directly into an xarray DataArray
    # (xarray chunks/streams directly from Azure Blob Storage COGs)
    xarray_ds = odc.stac.load(
        items,
        bands=["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12", "SCL"],
        crs=utm_crs,
        resolution=resolution,
        x=(minx - 20, maxx + 20),
        y=(miny - 20, maxy + 20),
        chunks={}
    ).load() # Materialize arrays into RAM

    if xarray_ds.sizes.get("time", 0) == 0:
        return np.nan
    
    # 5. Apply ESA baseline offset correction if needed (>4000 DN)
    for rawband in list(xarray_ds.data_vars): 
        xarray_ds[rawband] = xarray_ds[rawband].astype("float32")

    #6. Calculate Indices, Temporal Aggregate, and Spatial Aggregate 
    time_aggregate_mapping = {
            #Arg is not a good idea. We don't know what the start and end dates are and what they represent. 
            'min': lambda x: x.min(dim="time"),
            'mean': lambda x: x.mean(dim = "time") ,
            'median': lambda x: x.median(dim = 'time'),
            'max':lambda x: x.max(dim="time")
        }
    space_aggregate_mapping = {
    'mean': lambda x:  np.nanmean(x),
    'min': lambda x:  np.nanmin(x),
    'median': lambda x:  np.nanmedian(x),
    'max': lambda x:  np.nanmax(x)
            }
    transform = xarray_ds.odc.transform
    height, width = xarray_ds.sizes["y"], xarray_ds.sizes["x"]

    final_spatials_temporals_bands = {}
    for band in bands: 
        final_t_s = calculateSentinelIndex(xarray_ds[SENTINEL_INDEX_DEPENDENCIES[band]] , band)[band] #This saves memory ( I think?)
        # Band_t_s is now 3D 
        final_spatials_temporals = {} 
        for tempAg in temp_aggregations: 
            final_s  = time_aggregate_mapping[tempAg](final_t_s) 
            #Band_s is now 2D, and it needs to be rasterized first to remove pixels outside boundary. 
            # 7. Rasterizing each band. 
            mask = rasterio.features.rasterize(
                    [(geom, 1) for geom in event_utm.geometry],
                    out_shape=(height, width),
                    transform=transform,
                    fill=0,
                    dtype="uint8"
                )
            final_s_masked = np.where(mask == 1, final_s.values, np.nan)

            final_spatials = {}
            # 8. Spatial Aggregation
            for spatAg in spat_aggregations:
                final =  space_aggregate_mapping[spatAg](final_s_masked)
                # Finally, this is a single value. 
                final_spatials[spatAg] = float(final)
            final_spatials_temporals[tempAg] = final_spatials 

        final_spatials_temporals_bands[band] = final_spatials_temporals

    return final_spatials_temporals_bands


In [ ]:

SENTINEL_INDEX_DEPENDENCIES = {
    "NDVI": ["B04", "B08"],
    "NRFI": ["B03", "B11"],
    "NRFI2": ["B03", "B12"],
    "NIR": ["B08"],
    "RENDVI": ["B05", "B8A"],
    "NDRE": ["B05", "B08"],
    "NDWI": ["B08", "B11"],
    "NDYI": ["B02", "B03"],
    "YI": ["B03", "B04"],
    "YI2": ["B02", "B03", "B04"],
    "YFI": ["B03", "B05"],
    "SYR": ["B02", "B03", "B04"],
    "NDTI": ["B11", "B12"],
    "STI": ["B11", "B12"],
    "NDI7": ["B8A", "B12"],
    "SRNDI": ["B04", "B12"],
    "NDI5": ["B8A", "B11"],
    "NDSVI": ["B04", "B11"],
    "MCRC": ["B03", "B11"],
    "BAI": ["B04", "B08", "B12"],
    "BAI_HLS": ["B04", "B8A", "B12"],
    "VH": [],
    "VV": [],
    "VVVH": ["VH", "VV"],
}

kind = 'Sentinel2' # Kinda useless now, only Sentinel 2 in VM
bands = ['NDVI','NDTI','STI']
padding = 5
tempAg = ['min','mean','median']
spatAg = ['mean','median']
max_cloud_cover = 90

input_bands = []
for band in bands: 
    input_bands += SENTINEL_INDEX_DEPENDENCIES[band]
input_bands = list(set(input_bands))

In [ ]:

RUN_ACQUIRE_PATH = Path("/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/run_acquire.py")  
TEMP_DIR = Path("/home/azureuser/EBRD_Tillage/data/temp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DATA_PATH= Path("/home/azureuser/EBRD_Tillage/data/input/EBRD_merged_classified.shp")  

# Global Variables 
stratified_limitation = 10
buffer_days = 5
resolution = 10
target_to_use = 'alex_binary' # Options -- baseline (Classifica column), alex ,[ NOT IMPLEMENTED  binary_alex]  
## Alex sustainability Score 
#### This is currently approximate. In depth feature extraction with given depth, turnover etc need to be considered. 


kind = 'Sentinel2' # Kinda useless now, only Sentinel 2 in VM
bands = ['NDVI','NDTI','STI']
padding = 5
tempAg = ['min','mean','median']
spatAg = ['mean','median']
max_cloud_cover = 90

input_bands = []
for band in bands: 
    input_bands += SENTINEL_INDEX_DEPENDENCIES[band]
input_bands = list(set(input_bands))

NameError: name 'Path' is not defined

In [ ]:
# ==============================================================================
# 1. SETUP & PATH CONFIGURATION
# ==============================================================================
print(f"Reading shapefile from: {INPUT_DATA_PATH}")
gdf = gpd.read_file(INPUT_DATA_PATH).drop(columns = ['index'])
gdf=gdf.to_crs("EPSG:4326")
# Quick check
print(f"Successfully loaded {len(gdf)} rows.")
gdf = gdf.rename(columns={'Beginning':'start_date',"End date":'end_date'})
print(gdf.columns)
gdf["start_date"] = pd.to_datetime(gdf["start_date"])
gdf["end_date"] = pd.to_datetime(gdf["end_date"])


if(target_to_use == 'baseline'):
    gdf = classify_tillage(gdf,target_to_use)
    gdf = gdf[(gdf.Classifica == 'Conservation') | (gdf.Classifica == 'Conventional')]
    if(stratified_limitation >0): 
        good = gdf[gdf.Classifica == 'Conservation' ].sample(stratified_limitation)
        bad = gdf[gdf.Classifica == 'Conventional'].sample(stratified_limitation)
        gdf = pd.concat([good,bad],ignore_index=True).reset_index() 
elif(target_to_use == 'alex_binary'):
    gdf = classify_tillage(gdf,target_to_use)
    #gdf['target'] = [SUSTAINABILITY_CONVERSION[val] for val in gdf.]
    if(stratified_limitation>0): 
        lis = []
        for name,group in gdf.groupby('Target'): 
            if(group.shape[0] < stratified_limitation): 
                lis.append(group)
            else:
                lis.append(group.sample(stratified_limitation))
            print(sum(df.shape[0] for df in lis))
        gdf = pd.concat(lis).reset_index(drop=True)
print(f"Filtered to {len(gdf)} rows.")

Reading shapefile from: /home/azureuser/EBRD_Tillage/data/input/EBRD_merged_classified.shp
Successfully loaded 5746 rows.
Index(['fid', 'Operation', 'Group of o', 'Machinery', 'Machine', 'start_date',
       'end_date', 'Field ID', 'Crop', 'Year', 'Area, ha', 'operation_',
       'source_fil', 'FieldAreaH', 'Completion', 'Classifica', 'geometry'],
      dtype='str')
10
20
30
40
50
Filtered to 50 rows.


In [ ]:
# ==============================================================================
# 2. MAIN BATCH EXECUTION PIPELINE -- NEW
# ==============================================================================
# Connect to Planetary Computer STAC Catalog ONCE
catalog_client = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)
sample_gdf = gdf.sample()
utm_crs = get_utm_crs(sample_gdf)


print(f"Starting in-memory processing for {len(gdf)} tillage events...")



# Run in-memory extraction over events
band_results = []
for idx, row in gdf.iterrows():
    print(f"Processing Event {idx+1}/{len(gdf)}: {row['fid']} (TargetValue - {row['Target']})...", end="")
    get_sentinel2_STAC( 
        row,
        catalog_client,
        




    )
    def compute_event_band_in_memory( 
    row: pd.Series,
    
    bands :list,
    temp_aggregations :list,
    spat_aggregations:list,
    
    buffer_days: int = 10,
    max_cloud_cover: int = 90,
    resolution: float = 10.0,
        geometry: shapely.geometry, 
        
)
    final = compute_event_band_in_memory(row, catalog_client,bands = ['NDVI','NDTI'], temp_aggregations = ['min','mean'] , spat_aggregations=['mean','median'], buffer_days=10)

    geom_4326 = row.geometry
    # 1. Date window with buffer
    #start_dt = (pd.to_datetime(row["start_date"]) - pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
    start_dt = pd.to_datetime(row["end_date"]).strftime("%Y-%m-%d")
    end_dt = (pd.to_datetime(row["end_date"]) + pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
    
    try:
        band_val = compute_event_band_in_memory(row, catalog_client,band, buffer_days=10,tempAg='min',spatAg = 'mean')
        band_results.append(band_val)
        print(f" Done -> {band}: {band_val:.4f}")
    except Exception as e:
        print(f" Failed -> Error: {e}")
        band_results.append(np.nan)

gdf["summary_band"] = band_results

# Filter out failed/empty retrievals
analysis_df = gdf.dropna(subset=["summary_band"]).copy()

NameError: name 'ds' is not defined

In [ ]:
# ==============================================================================
# 2. MAIN BATCH EXECUTION PIPELINE
# ==============================================================================
# Connect to Planetary Computer STAC Catalog ONCE
catalog_client = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

print(f"Starting in-memory processing for {len(gdf)} tillage events...")

# Run in-memory extraction over events
band_results = []
for idx, row in gdf.iterrows():
    print(f"Processing Event {idx+1}/{len(gdf)}: {row['fid']} (TargetValue - {row['Target']})...", end="")
    try:
        band_val = compute_event_band_in_memory(row, catalog_client,band, buffer_days=10,tempAg='min',spatAg = 'mean')
        band_results.append(band_val)
        print(f" Done -> {band}: {band_val:.4f}")
    except Exception as e:
        print(f" Failed -> Error: {e}")
        band_results.append(np.nan)

gdf["summary_band"] = band_results

# Filter out failed/empty retrievals
analysis_df = gdf.dropna(subset=["summary_band"]).copy()

In [ ]:
# ==============================================================================
# 3. THRESHOLD OPTIMIZATION & VISUALIZATION
# ==============================================================================

if(target_to_use == 'baseline'):
    # Encode target: Conventional = 0, Conservation = 1
    analysis_df["binary_target"] = (analysis_df["Target"].str.lower() == "conservation").astype(int)
elif(target_to_use == 'alex_binary'): 
    #minval = min(analysis_df.Target)
    val = analysis_df.Target.mean()
    analysis_df["binary_target"] =  (analysis_df['Target'] > val).astype(int)
else: 
    raise NotImplementedError 
y_true = analysis_df["binary_target"].values
scores = analysis_df["summary_band"].values

candidate_thresholds = np.linspace(scores.min(), scores.max(), 200)

best_f1_unidirectional, best_acc_unidirectional, best_threshold = -1.0, -1.0, candidate_thresholds[0]
f1_scores, acc_scores = [], []

for t in candidate_thresholds:
    
    if(target_to_use in ['baseline', 'alex_binary']):
        y_pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
    elif(target_to_use == 'alex'): 

        f1 = f1_score(y_true, y_pred, zero_division=0, average = 'micro') 
    else:
        NotImplementedError
    acc = accuracy_score(y_true, y_pred) 
    f1 = f1 if f1>0.5 else 1-f1 
    acc = acc if acc>0.5 else 1-acc
    
    f1_scores.append(f1)
    acc_scores.append(acc)
    
    #if f1 > best_f1 or (f1 == best_f1 and acc > best_acc):
    if acc > best_acc_unidirectional or (f1 > best_f1_unidirectional and acc == best_acc_unidirectional):
        best_f1_unidirectional, best_acc_unidirectional, best_threshold = f1, acc, t

print("\n================ OPTIMIZATION RESULTS ================")
print(f"Optimal {band} Threshold: {best_threshold:.4f}")
print(f"Maximized F1-Score (Unidirectional):     {best_f1_unidirectional:.4f}")
print(f"Overall Accuracy (Unidirectional):       {best_acc_unidirectional:.4f}")
print("======================================================")

# Plot distributions and optimization curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

conv_scores = analysis_df[analysis_df["binary_target"] == 0]["summary_band"]
cons_scores = analysis_df[analysis_df["binary_target"] == 1]["summary_band"]

label0 , label1 = np.unique(analysis_df['Target'].astype(str))
ax1.hist(conv_scores, bins=15, alpha=0.6, color="orangered", label=label0)
ax1.hist(cons_scores, bins=15, alpha=0.6, color="seagreen", label=label1)
ax1.axvline(best_threshold, color="black", linestyle="--", linewidth=2, label=f"Threshold ({best_threshold:.3f})")
ax1.set_xlabel(f"Summary {band} (Temporal Min → Spatial Mean)")
ax1.set_ylabel("Count")
ax1.set_title(f"{band} Distribution by Tillage Type")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(candidate_thresholds, f1_scores, color="purple", label="F1-Score", linewidth=2)
ax2.plot(candidate_thresholds, acc_scores, color="dodgerblue", label="Accuracy", linewidth=2)
ax2.axvline(best_threshold, color="black", linestyle="--", linewidth=1.5, label="Optimal Cutoff")
ax2.set_xlabel("Candidate Threshold")
ax2.set_ylabel("Score")
ax2.set_title("F1 / Accuracy Optimization Curve")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
1/0

In [ ]:
# ==============================================================================
# 3. THRESHOLD OPTIMIZATION & VISUALIZATION -- COPY
# ==============================================================================

if(target_to_use == 'baseline'):
    # Encode target: Conventional = 0, Conservation = 1
    analysis_df["binary_target"] = (analysis_df["Target"].str.lower() == "conservation").astype(int)
elif(target_to_use == 'alex'): 
    minval = min(analysis_df.Target)
    analysis_df["binary_target"] =  analysis_df['binary_target'] - minval  
else: 
    raise NotImplementedError 
y_true = analysis_df["binary_target"].values
scores = analysis_df["summary_band"].values

candidate_thresholds = np.linspace(scores.min(), scores.max(), 200)

best_f1, best_acc, best_threshold = -1.0, -1.0, candidate_thresholds[0]
f1_scores, acc_scores = [], []

for t in candidate_thresholds:
    y_pred = (scores >= t).astype(int)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    
    f1_scores.append(f1)
    acc_scores.append(acc)
    
    #if f1 > best_f1 or (f1 == best_f1 and acc > best_acc):
    if acc > best_acc or (f1 > best_f1 and acc == best_acc):
        best_f1, best_acc, best_threshold = f1, acc, t

print("\n================ OPTIMIZATION RESULTS ================")
print(f"Optimal {band} Threshold: {best_threshold:.4f}")
print(f"Maximized F1-Score:     {best_f1:.4f}")
print(f"Overall Accuracy:       {best_acc:.4f}")
print("======================================================")

# Plot distributions and optimization curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

conv_scores = analysis_df[analysis_df["binary_target"] == 0]["summary_band"]
cons_scores = analysis_df[analysis_df["binary_target"] == 1]["summary_band"]

ax1.hist(conv_scores, bins=15, alpha=0.6, color="orangered", label="Conventional")
ax1.hist(cons_scores, bins=15, alpha=0.6, color="seagreen", label="Conservation")
ax1.axvline(best_threshold, color="black", linestyle="--", linewidth=2, label=f"Threshold ({best_threshold:.3f})")
ax1.set_xlabel(f"Summary {band} (Temporal Min → Spatial Mean)")
ax1.set_ylabel("Count")
ax1.set_title(f"{band} Distribution by Tillage Type")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(candidate_thresholds, f1_scores, color="purple", label="F1-Score", linewidth=2)
ax2.plot(candidate_thresholds, acc_scores, color="dodgerblue", label="Accuracy", linewidth=2)
ax2.axvline(best_threshold, color="black", linestyle="--", linewidth=1.5, label="Optimal Cutoff")
ax2.set_xlabel("Candidate Threshold")
ax2.set_ylabel("Score")
ax2.set_title("F1 / Accuracy Optimization Curve")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()